In [1]:
import pandas as pd
import sys
import os
from pathlib import Path
import numpy as np
import statsmodels.api as sm
from scipy.stats import spearmanr, t

from scipy.stats import t as t_dist

# Lấy đường dẫn thư mục gốc (nơi chứa src/)
project_root = Path.cwd().parent  # Đi lên 1 cấp từ notebook_file/
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


df = pd.read_parquet("../data/processed/B_factor_construction.parquet")

df

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d,fwd_1d,fwd_5d,fwd_14d
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,0.230133,0.473426,1.166085,0.264551,-0.374500,-0.483342,-1.207511,-1.162525,-1.333472,-0.707390,0.004484,-1.152049,-1.279244,0.003372,0.019895,0.227973
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,0.190681,0.716782,0.899746,0.424706,-0.449360,-0.449496,-0.961410,-0.921146,-0.631311,-0.026073,0.003514,-1.136343,-1.252158,0.011316,0.055735,0.257275
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,0.090847,0.191346,0.455889,-0.068698,-0.765769,-0.828524,0.864584,-0.075388,-0.642522,-1.009617,-0.000590,-0.936310,-1.048271,0.027662,0.016399,0.126829
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.976768,-0.683009,-0.747885,1.001352,-0.731231,-0.825763,0.357615,0.717912,0.249583,-0.571819,-0.000142,-0.853872,-0.953418,0.016380,0.030426,0.203359
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,0.557226,0.257101,0.310989,0.279063,-1.232723,-0.742282,-1.212615,-0.569293,-0.352224,0.303731,-0.007744,-0.952727,-1.058538,0.003268,0.061491,0.212283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258042,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003923,NaN,NaN,0.024672,0.105572,0.146774
258043,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000985,NaN,NaN,0.012474,0.084218,0.092330
258044,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007752,NaN,NaN,0.017225,0.055387,0.042519
258045,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.015152,NaN,NaN,0.073563,0.161755,0.094616


In [2]:
col_forward_return = ["fwd_1d","fwd_5d","fwd_14d"]

col_factor = ["z_momentum_7d", "z_momentum_14d", "z_momentum_30d", "z_momentum_90d", "z_reversal_1d", "z_reversal_3d", "z_vol_7d",
              "z_vol_14d", "z_vol_30d", "z_vol_of_vol_14d", "z_amihud_14d", "z_amihud_30d"]

In [ ]:
df["timestamp"].nunique()

1096

# Phần C : Factor Validation

- Mục tiêu lớn nhất của phần C, là đánh giá các chỉ số thống kê của các factor ta đã tạo từ phần B
- Sau khi có các chỉ số thống kê, ta sẽ chỉ lựa chọn các factor có ý nghĩa và mang tính độc lập với nhau

## C1 — Daily Information Coefficient

- Xây dựng ra một dataframe với từng cột lần lượt là timestamp, factor, horizon, coeff_spearman   
- Ứng với ngày thứ t, sẽ là coeff spearman giữa factor và một horizon, mỗi factor trong 1 ngày tương ứng 3 dòng trong dataframe


In [4]:
df[df["timestamp"] == "2023-01-01 00:00:00+00:00"].head()

,timestamp,symbol,in_universe,z_momentum_7d,z_momentum_14d,z_momentum_30d,z_momentum_90d,z_reversal_1d,z_reversal_3d,z_vol_7d,z_vol_14d,z_vol_30d,z_vol_of_vol_14d,log_return,z_amihud_14d,z_amihud_30d,fwd_1d,fwd_5d,fwd_14d
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,0.230133,0.473426,1.166085,0.264551,-0.374500,-0.483342,-1.207511,-1.162525,-1.333472,-0.707390,0.004484,-1.152049,-1.279244,0.003372,0.019895,0.227973
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,0.190681,0.716782,0.899746,0.424706,-0.449360,-0.449496,-0.961410,-0.921146,-0.631311,-0.026073,0.003514,-1.136343,-1.252158,0.011316,0.055735,0.257275
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,0.090847,0.191346,0.455889,-0.068698,-0.765769,-0.828524,0.864584,-0.075388,-0.642522,-1.009617,-0.000590,-0.936310,-1.048271,0.027662,0.016399,0.126829
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.976768,-0.683009,-0.747885,1.001352,-0.731231,-0.825763,0.357615,0.717912,0.249583,-0.571819,-0.000142,-0.853872,-0.953418,0.016380,0.030426,0.203359
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,0.557226,0.257101,0.310989,0.279063,-1.232723,-0.742282,-1.212615,-0.569293,-0.352224,0.303731,-0.007744,-0.952727,-1.058538,0.003268,0.061491,0.212283


In [5]:
"""
Mục tiêu của hàm này là tạo ra một dataframe
Tính spearman coeff cho từng foward_return (fwd_1,5,14d)
Ứng với ngày thứ t, với từng factor, ta sẽ tính coeff cho giữa factor và 3 biến foward_return
Với mỗi ngày, mỗi biến sẽ có 3 dòng, vì có 12 factor nên sẽ có 12*3=36 dòng/ ngày và có khoảng 1096*36 dòng 

input là df, col_factor (danh sách các cột factor), col_foward_return (danh sách các cột foward) 
"""

def build_daily_ic(df, col_factor, col_forward_return):
    in_universe = df[df["in_universe"] == 1].copy()
    
    groups = in_universe.groupby("timestamp")
    result = []
    
    for date, group in groups:
        factor_data = group[col_factor]
        foward_data = group[col_forward_return]
        
        for i, factor in enumerate(col_factor):
            factor_value = factor_data.loc[:,factor]
            
            for j, foward in enumerate(col_forward_return):
                foward_value = foward_data.loc[:,foward]
                
                corr, p_value = spearmanr(factor_value, foward_value)
                result.append({
                    "timestamp": date,
                    "factor_id": factor,
                    "horizon":foward,
                    "ic_spearmanr": corr  
                })
                
                
    ic_daily = pd.DataFrame(result)
    return ic_daily

daily_ic = build_daily_ic(df, col_factor, col_forward_return)
daily_ic

,timestamp,factor_id,horizon,ic_spearmanr
0,2023-01-01 00:00:00+00:00,z_momentum_7d,fwd_1d,-0.204545
1,2023-01-01 00:00:00+00:00,z_momentum_7d,fwd_5d,-0.445652
2,2023-01-01 00:00:00+00:00,z_momentum_7d,fwd_14d,-0.427866
3,2023-01-01 00:00:00+00:00,z_momentum_14d,fwd_1d,-0.352767
4,2023-01-01 00:00:00+00:00,z_momentum_14d,fwd_5d,-0.361660
...,...,...,...,...
39451,2025-12-31 00:00:00+00:00,z_amihud_14d,fwd_5d,0.354902
39452,2025-12-31 00:00:00+00:00,z_amihud_14d,fwd_14d,0.403922
39453,2025-12-31 00:00:00+00:00,z_amihud_30d,fwd_1d,0.432773
39454,2025-12-31 00:00:00+00:00,z_amihud_30d,fwd_5d,0.321289


## C2  — IC Inference: Mean, Newey-West t-stat
Sau khi có spearman theo ngày của từng factor ứng với mỗi horizon, ta xây dựng thêm bảng thống kê thể hiện chỉ số ic_mean, ic_std, ic_ir:  
- ic_mean: tương ứng giá trị trung bình của coeff_spearman của 1 factor với 1 horizon trong toàn bộ khoảng thời gian  
- ic_std: tương tự ic_mean nhưng cho ic_std  
- ic_ir: ic_mean / ic_std   
- Và kiểm định thống kê ý nghĩa ic_mean  

In [21]:
""" 
Mục tiêu của hàm này là tạo ra bảng thống kê của ic_daily (IC: information coeff)
Những chỉ số quan trọng là: ic_mean, ic_std, ic_ir = ic_mean/ic_std, ic_t_stat
"""
def build_statistics_ic(daily_ic):
    df = daily_ic.copy()
    
    temp = df.groupby(["factor_id","horizon"]).agg(
        ic_mean= ("ic_spearmanr", 'mean'),
        ic_std = ("ic_spearmanr", "std")
    )
    
    temp["ic_ir"] = temp["ic_mean"] / temp["ic_std"]
    return temp


def calculate_nw_statistics(daily_ic, lag_map=None):

    if lag_map is None:
        lag_map = {
            "fwd_1d": 0,
            "fwd_5d": 4,
            "fwd_14d": 13
        }

    results = []

    for (factor_id, foward_id), group in daily_ic.groupby(
        ["factor_id", "horizon"]
    ):

        ic = (
            group["ic_spearmanr"]
            .dropna()
            .astype(float)
            .values
        )

        n_days = len(ic)

        if n_days < 2:
            results.append({
                "factor_id": factor_id,
                "horizon": foward_id,
                "nw_lag": np.nan,
                "nw_se": np.nan,
                "nw_t_stat": np.nan,
                "n_days": n_days
            })
            continue

        lag = min(
            lag_map[foward_id],
            n_days - 1
        )

        X = np.ones((n_days, 1))

        model = sm.OLS(ic, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": lag}
        )

        results.append({
            "factor_id": factor_id,
            "horizon": foward_id,
            "nw_lag": lag,
            "nw_se": model.bse[0],
            "nw_t_stat": model.tvalues[0],
            "n_days": n_days
        })

    return pd.DataFrame(results)



def add_p_value(nw_stats):

    df = nw_stats.copy()

    df["p_value"] = (
        2 * t.sf(
            np.abs(df["nw_t_stat"]),
            df["n_days"] - 1
        )
    )

    return df


def build_ic_summary(daily_ic):

    statistics = (
        build_statistics_ic(daily_ic)
        .reset_index()
    )

    nw_stats = calculate_nw_statistics(daily_ic)

    nw_stats = add_p_value(nw_stats)

    summary = (
        statistics
        .merge(
            nw_stats,
            on=["factor_id", "horizon"],
            how="left"
        )
    )
    
    def sig_star(p):
        if p < 0.01:
            return 'alpha 1%'
        elif p < 0.05:
            return 'alpha 5%'
        else:
            return 'not significant'
    summary["sig"] = summary["p_value"].apply(lambda x: sig_star(x))

    return summary



ic_statistics = build_ic_summary(daily_ic)
ic_statistics

,factor_id,horizon,ic_mean,ic_std,ic_ir,nw_lag,nw_se,nw_t_stat,n_days,p_value,sig
0,z_amihud_14d,fwd_14d,-0.139387,0.270425,-0.515436,13,0.025971,-5.367088,805,1.047547e-07,alpha 1%
1,z_amihud_14d,fwd_1d,-0.061571,0.290027,-0.212295,0,0.010216,-6.027088,805,2.541274e-09,alpha 1%
2,z_amihud_14d,fwd_5d,-0.101050,0.290286,-0.348105,4,0.017740,-5.696162,805,1.718678e-08,alpha 1%
3,z_amihud_30d,fwd_14d,-0.159735,0.278458,-0.573640,13,0.032554,-4.906792,585,1.202373e-06,alpha 1%
4,z_amihud_30d,fwd_1d,-0.072090,0.296811,-0.242881,0,0.012261,-5.879529,585,6.924880e-09,alpha 1%
5,z_amihud_30d,fwd_5d,-0.108743,0.293309,-0.370746,4,0.021154,-5.140525,585,3.742880e-07,alpha 1%
6,z_momentum_14d,fwd_14d,0.048927,0.257932,0.189690,13,0.022912,2.135456,805,3.302535e-02,alpha 5%
7,z_momentum_14d,fwd_1d,-0.013561,0.272274,-0.049806,0,0.009590,-1.414012,805,1.577454e-01,not significant
8,z_momentum_14d,fwd_5d,0.010104,0.263323,0.038371,4,0.014993,0.673913,805,5.005606e-01,not significant
9,z_momentum_30d,fwd_14d,-0.003142,0.271403,-0.011577,13,0.028821,-0.109017,585,9.132264e-01,not significant


## C3 — Univariate Fama–MacBeth

Phần này trả lời cho câu hỏi: theo C2 thì những factor có quan hệ với foward return thì ở phần C3 này sẽ trả lời nếu có thì sẽ biểu
diễn thành quan hệ tuyến tính thế nào  

- Ta xây dựng mô hình hồi quy cho từng ngày, ứng với mỗi ngày t chọn ra các coin hiện có trong UNIVERSE và tạo mô hình hồi quy dựa vào factor và target là horizon (gọi hệ số của factor là beta i)  

- Sau đó, trên toàn bộ khoảng thời gian ta tính mean của beta i  
- Đồng thời kiểm định giá trị beta đó có ý nghĩa thống kê hay không  


In [ ]:

""" 
Ta thực hiện chạy hồi quy cho toàn bộ factor cho từng ngày, ứng với mỗi factor ta kiểm tra độ tuyến tính với từng foward return và mối quan hệ đó có 
ý nghĩa thống kê hay không.
Chỉ thực hiện trên nhãn in_universe == 1
Ví dụ ngày t, ta muốn đo độ tuyến tính giũa momentum_7d với fwd_5d và cần biết có ý nghĩa thống kê hay không.
"""

def build_regression_daily(df, col_factor, col_forward_return):
    in_universe = df[df["in_universe"] == 1].copy()
    
    results = []
    for factor in col_factor:
        for forward in col_forward_return:
            beta_series= []
            for date, group in in_universe.groupby("timestamp"):
                group_regression = group.dropna(subset=[factor, forward])
                x = sm.add_constant(group_regression[factor])
                y = group_regression[forward]
                
                model = sm.OLS(y, x).fit()
                beta_t = model.params.iloc[1]
                alpha_t = model.params.iloc[0]
                
                beta_series.append({
                    "date": date,
                    "beta_t": beta_t,
                    "alpha_t": alpha_t,
                    "n_obs": len(group_regression)
                })                    
            beta_df = pd.DataFrame(beta_series)
            
            
            
            n_days = len(beta_df)
            beta_mean = beta_df['beta_t'].mean()
            beta_std = beta_df['beta_t'].std()
            alpha_mean = beta_df['alpha_t'].mean()
            
            # NEWEY-WEST T-STAT
            # Xác định số lag dựa trên forward horizon
            if forward == 'fwd_1d':
                maxlag = 0
            elif forward == 'fwd_5d':
                maxlag = 4
            elif forward == 'fwd_14d':
                maxlag = 13
            else:
                maxlag = 0
            
            y_nw = beta_df['beta_t'].values
            X_nw = np.ones((n_days, 1))
            
            try:
                model_nw = sm.OLS(y_nw, X_nw).fit(
                    cov_type='HAC',
                    cov_kwds={'maxlags': maxlag}
                )
                nw_se = model_nw.bse[0]
                nw_t_stat = model_nw.tvalues[0]
            except:
                nw_se = beta_std / np.sqrt(n_days)
                nw_t_stat = beta_mean / nw_se
            
            # p-value (2-tailed)
            p_value = 2 * t.sf(abs(nw_t_stat), n_days - 1)
            
            results.append({
                'factor_id': factor,
                'horizon': forward,
                'beta_mean': beta_mean,
                'beta_std': beta_std,
                'nw_se': nw_se,
                'nw_t_stat': nw_t_stat,
                'p_value': p_value,
                'alpha_mean': alpha_mean,
                'n_days': n_days,
                'avg_n_obs': beta_df['n_obs'].mean()
            })
    
    fm_summary = pd.DataFrame(results)
    
    def sig_star(p):
        if p < 0.01:
            return 'alpha 1%'
        elif p < 0.05:
            return 'alpha 5%'
        else:
            return 'not significant'
    
    fm_summary['sig'] = fm_summary['p_value'].apply(sig_star)
    
    # Sắp xếp
    horizon_order = {'fwd_1d': 0, 'fwd_5d': 1, 'fwd_14d': 2}
    fm_summary['horizon_rank'] = fm_summary['horizon'].map(horizon_order)
    fm_summary = fm_summary.sort_values(['factor_id', 'horizon_rank'])
    fm_summary = fm_summary.drop(columns=['horizon_rank'])
    
    return fm_summary.reset_index(drop=True)

build_regression_daily(df, col_factor, col_forward_return)

,factor_id,horizon,beta_mean,beta_std,nw_se,nw_t_stat,p_value,alpha_mean,n_days,avg_n_obs,sig
0,z_amihud_14d,fwd_1d,-0.000897,0.010238,0.000309,-2.901533,0.003788,-0.000737,1096,36.763686,alpha 1%
1,z_amihud_14d,fwd_5d,-0.004532,0.021257,0.001157,-3.917055,0.000095,-0.003413,1096,36.763686,alpha 1%
2,z_amihud_14d,fwd_14d,-0.013487,0.032877,0.002770,-4.868763,0.000001,-0.010474,1096,36.763686,alpha 1%
3,z_amihud_30d,fwd_1d,-0.001102,0.011754,0.000355,-3.105622,0.001948,-0.000742,1096,36.428832,alpha 1%
4,z_amihud_30d,fwd_5d,-0.005147,0.024207,0.001312,-3.923241,0.000093,-0.003467,1096,36.428832,alpha 1%
5,z_amihud_30d,fwd_14d,-0.014426,0.035560,0.002989,-4.826565,0.000002,-0.010775,1096,36.428832,alpha 1%
6,z_momentum_14d,fwd_1d,0.000048,0.012804,0.000387,0.125031,0.900522,-0.000737,1096,36.763686,not significant
7,z_momentum_14d,fwd_5d,0.001751,0.024960,0.001255,1.395858,0.163040,-0.003413,1096,36.763686,not significant
8,z_momentum_14d,fwd_14d,0.006495,0.035320,0.002556,2.541080,0.011188,-0.010474,1096,36.763686,alpha 5%
9,z_momentum_30d,fwd_1d,-0.000210,0.013109,0.000396,-0.531560,0.595138,-0.000742,1096,36.428832,not significant


## C4 — Quintile Portfolio Spread


Vì ta sẽ thực hiện mô hình tuyến tính ở các phần sau nên việc xác định dấu mong muốn của 1 factor với 1 horizon là cần thiết  

Ta kỳ vọng dấu giữa spread_mean, beta_mean, ic_mean cần chung dấu (vì ta mong muốn diễn giải được quan hệ tuyến tính theo kỳ vọng):  
- Nghĩa là khi spread_mean âm giữa một factor x một horizon, đồng nghĩa z_score của factor đó càng lớn thì horizon đó càng nhỏ  
- Nếu spread_mean dương thì z_socre của factor đó càng lớn thì horizon càng lớn  
- beta_mean dương nghĩa là xu hướng giữa factor và horizon, factor càng lớn thì horizon cũng có xu hướng tăng theo  
     
Đồng thời thêm điều kiện chỉ chọn factor x horizon mà có spread_mean đủ độ lớn để bù phần chi phí khác (ví dụ chi phí giao dịch)  


In [8]:
""" 
Ứng với ngày t, của factor i, ta chia giá trị của factor i đang xét thành 3 nhóm High, Low,  Mid
Sắp xếp theo chiều giá trị factor tăng dần, và gán xem coin nào thuộc nhóm nào
Spread = average_foward_return (nhóm Hight) - average_forward_return (nhóm Low) 

Sau đó tổng hợp qua toàn bộ thời gian, tính spread_mean, spread_Std và ý nghĩa thống kê
"""
def build_day_portfolio_spread(df, col_factor, col_forward_return, n_group = 3):
    in_universe = df[df["in_universe"] == 1].copy()

    results = []
    for date, group in in_universe.groupby("timestamp"):
        group = group.dropna()
        n_coins = len(group)
        for factor in col_factor:
            
            factor_group = group.sort_values(factor)
            
            factor_group["group"] = pd.qcut(np.arange(n_coins), q = n_group, labels= range(n_group))
            
            for forward_return in col_forward_return:
                group_mean = factor_group.groupby("group")[forward_return].mean()
                
                avg_low = group_mean.iloc[0]
                avg_mid = group_mean.iloc[1]
                avg_high = group_mean.iloc[2]
                
                    
                spread = avg_high - avg_low
                
                results.append({
                    'date': date,
                    'factor_id': factor,
                    'forward_return': forward_return,
                    'spread': spread,
                    'avg_return_low': avg_low,
                    'avg_return_mid': avg_mid,
                    'avg_return_high': avg_high
                })
    return pd.DataFrame(results)

day_portfolio_spread = build_day_portfolio_spread(df, col_factor, col_forward_return)
day_portfolio_spread

,date,factor_id,forward_return,spread,avg_return_low,avg_return_mid,avg_return_high
0,2023-01-01 00:00:00+00:00,z_momentum_7d,fwd_1d,-0.014894,0.038712,0.014325,0.023818
1,2023-01-01 00:00:00+00:00,z_momentum_7d,fwd_5d,-0.076600,0.120836,0.037901,0.044237
2,2023-01-01 00:00:00+00:00,z_momentum_7d,fwd_14d,-0.217248,0.433950,0.236738,0.216702
3,2023-01-01 00:00:00+00:00,z_momentum_14d,fwd_1d,-0.022711,0.042591,0.013830,0.019880
4,2023-01-01 00:00:00+00:00,z_momentum_14d,fwd_5d,-0.080215,0.118269,0.047018,0.038054
...,...,...,...,...,...,...,...
39451,2025-12-31 00:00:00+00:00,z_amihud_14d,fwd_5d,0.103960,0.094240,0.188033,0.198199
39452,2025-12-31 00:00:00+00:00,z_amihud_14d,fwd_14d,0.159049,0.092934,0.163274,0.251984
39453,2025-12-31 00:00:00+00:00,z_amihud_30d,fwd_1d,0.049826,0.018898,0.061903,0.068724
39454,2025-12-31 00:00:00+00:00,z_amihud_30d,fwd_5d,0.103960,0.094240,0.188033,0.198199


In [9]:
def build_portfolio(day_portfolio_spread):
    
    horizon_lag_map = {
            'fwd_1d': 0,
            'fwd_5d': 4,
            'fwd_14d': 13
        }
    
    df = day_portfolio_spread.copy()
    
    results = []
    
    for (factor_id, horizon), group in df.groupby(['factor_id', 'forward_return']):
        group = group.sort_values('date')
        spread_series = group['spread'].dropna()
        n_days = len(spread_series)
        
        spread_mean = spread_series.mean()
        spread_std = spread_series.std()
        
        lag = min(horizon_lag_map.get(horizon, 0), n_days - 1)
        x = np.ones((n_days, 1))
        
        model = sm.OLS(spread_series.values, x).fit(
                cov_type='HAC',
                cov_kwds={'maxlags': lag}
            )
        nw_se = model.bse[0]
        nw_t_stat = model.tvalues[0]
        
        p_value = 2 * t.sf(abs(nw_t_stat), n_days - 1)
        
        
        results.append({
            'factor_id': factor_id,
            'horizon': horizon,
            'spread_mean': spread_mean,
            'spread_std': spread_std,
            'nw_se': nw_se,
            'nw_t_stat': nw_t_stat,
            'p_value': p_value
        })
    
    summary = pd.DataFrame(results)
    
    def sig_star(p):
        if p < 0.01:
            return "alpha 1%"
        elif p < 0.05:
            return "alpha 5%"
        else:
            return "not significant"
    
    summary['sig'] = summary['p_value'].apply(sig_star)
    

    horizon_order = {'fwd_1d': 0, 'fwd_5d': 1, 'fwd_14d': 2}
    summary['horizon_rank'] = summary['horizon'].map(horizon_order)
    summary = summary.sort_values(['factor_id', 'horizon_rank'])
    summary = summary.drop(columns=['horizon_rank'])
    
    return summary.reset_index(drop=True)

build_portfolio(day_portfolio_spread)

,factor_id,horizon,spread_mean,spread_std,nw_se,nw_t_stat,p_value,sig
0,z_amihud_14d,fwd_1d,-0.001911,0.022545,0.000681,-2.807070,0.005088,alpha 1%
1,z_amihud_14d,fwd_5d,-0.009236,0.047540,0.002578,-3.582839,0.000355,alpha 1%
2,z_amihud_14d,fwd_14d,-0.027418,0.075206,0.006606,-4.150476,0.000036,alpha 1%
3,z_amihud_30d,fwd_1d,-0.001881,0.023118,0.000698,-2.695400,0.007138,alpha 1%
4,z_amihud_30d,fwd_5d,-0.009176,0.050316,0.002764,-3.319709,0.000931,alpha 1%
5,z_amihud_30d,fwd_14d,-0.028051,0.078631,0.007007,-4.003082,0.000067,alpha 1%
6,z_momentum_14d,fwd_1d,0.000882,0.022576,0.000682,1.294651,0.195714,not significant
7,z_momentum_14d,fwd_5d,0.005320,0.047389,0.002430,2.189365,0.028781,alpha 5%
8,z_momentum_14d,fwd_14d,0.012755,0.070037,0.005278,2.416762,0.015822,alpha 5%
9,z_momentum_30d,fwd_1d,0.000489,0.023550,0.000711,0.687127,0.492148,not significant


In [10]:
def build_summary_factor(df, col_factor, col_forward_return):
    daily_ic = build_daily_ic(df, col_factor, col_forward_return)
    ic_summary = (build_ic_summary(daily_ic))[["factor_id","horizon","ic_mean","sig"]]
    regression_summary = (build_regression_daily(df, col_factor, col_forward_return))[["factor_id","horizon", "beta_mean","sig"]]
    day_portfolio_spread = build_day_portfolio_spread(df, col_factor, col_forward_return)
    portfolio_summary = (build_portfolio(day_portfolio_spread))[["factor_id","horizon","spread_mean","sig"]]
    
    result = (portfolio_summary.merge(ic_summary, on = ["factor_id","horizon"], how ="inner")).merge(
        regression_summary, on = ["factor_id","horizon"], how ="inner")
    
    return result

summary_factor = build_summary_factor(df, col_factor, col_forward_return)
summary_factor

,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig
0,z_amihud_14d,fwd_1d,-0.001911,alpha 1%,-0.061571,alpha 1%,-0.000897,alpha 1%
1,z_amihud_14d,fwd_5d,-0.009236,alpha 1%,-0.101050,alpha 1%,-0.004532,alpha 1%
2,z_amihud_14d,fwd_14d,-0.027418,alpha 1%,-0.139387,alpha 1%,-0.013487,alpha 1%
3,z_amihud_30d,fwd_1d,-0.001881,alpha 1%,-0.072090,alpha 1%,-0.001102,alpha 1%
4,z_amihud_30d,fwd_5d,-0.009176,alpha 1%,-0.108743,alpha 1%,-0.005147,alpha 1%
5,z_amihud_30d,fwd_14d,-0.028051,alpha 1%,-0.159735,alpha 1%,-0.014426,alpha 1%
6,z_momentum_14d,fwd_1d,0.000882,not significant,-0.013561,not significant,0.000048,not significant
7,z_momentum_14d,fwd_5d,0.005320,alpha 5%,0.010104,not significant,0.001751,not significant
8,z_momentum_14d,fwd_14d,0.012755,alpha 5%,0.048927,alpha 5%,0.006495,alpha 5%
9,z_momentum_30d,fwd_1d,0.000489,not significant,-0.020867,not significant,-0.000210,not significant


In [11]:
def cal_speard_mean_day(row):
    if row["horizon"] == "fwd_1d":
        return 1
    elif row["horizon"] == "fwd_5d":
        return 5
    else:
        return 14

summary_factor_1 = summary_factor.copy()
summary_factor_1["spread_mean_magnitude"] = summary_factor_1.apply(lambda x: abs(x["spread_mean"] *10000)/ cal_speard_mean_day(x) , axis = 1)
summary_factor_1



,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig,spread_mean_magnitude
0,z_amihud_14d,fwd_1d,-0.001911,alpha 1%,-0.061571,alpha 1%,-0.000897,alpha 1%,19.107714
1,z_amihud_14d,fwd_5d,-0.009236,alpha 1%,-0.101050,alpha 1%,-0.004532,alpha 1%,18.471505
2,z_amihud_14d,fwd_14d,-0.027418,alpha 1%,-0.139387,alpha 1%,-0.013487,alpha 1%,19.584141
3,z_amihud_30d,fwd_1d,-0.001881,alpha 1%,-0.072090,alpha 1%,-0.001102,alpha 1%,18.813110
4,z_amihud_30d,fwd_5d,-0.009176,alpha 1%,-0.108743,alpha 1%,-0.005147,alpha 1%,18.352976
5,z_amihud_30d,fwd_14d,-0.028051,alpha 1%,-0.159735,alpha 1%,-0.014426,alpha 1%,20.036652
6,z_momentum_14d,fwd_1d,0.000882,not significant,-0.013561,not significant,0.000048,not significant,8.824701
7,z_momentum_14d,fwd_5d,0.005320,alpha 5%,0.010104,not significant,0.001751,not significant,10.640660
8,z_momentum_14d,fwd_14d,0.012755,alpha 5%,0.048927,alpha 5%,0.006495,alpha 5%,9.110751
9,z_momentum_30d,fwd_1d,0.000489,not significant,-0.020867,not significant,-0.000210,not significant,4.885709


Thực hiện chọn factor theo hướng:
- Có ít nhất 2 kiểm định ý nghĩa alpha 1%  
- spread_mean, beta, ic_mean cần cùng dấu -> bắt quan hệ tuyến tính cho mô hình hồi quy phía sau  
- độ lớn của spread_mean đủ lớn để bù các chi phí giao dịch...  

In [12]:
def check_sign(x):
    if x > 0:
        return 1
    elif x< 0:
        return -1
    else: 
        return 0

# input là mỗi dòng trong bảng summary_factor
def check_quality_factor(row):
    sig_x = row["sig_x"] == "alpha 1%"
    sig_y = row["sig_y"] == "alpha 1%"
    sig = row["sig"] == "alpha 1%"
    n_sig = sum([sig_x, sig_y, sig]) # số kiểm định hợp lệ
    
    spread_mean_sign = check_sign(row["spread_mean"])
    ic_mean_sign = check_sign(row["ic_mean"])
    beta_mean_sign = check_sign(row["beta_mean"])
    
    sign = [spread_mean_sign, ic_mean_sign, beta_mean_sign]
    sign_consistence = 1 if (sum(sign) == 3 or sum(sign) == -3) else 0 # cùng chiều hay k
    
    is_spread_enough = row["spread_mean_magnitude"] >= 15
    
    if n_sig >= 2 and sign_consistence == 1 and is_spread_enough:
        return True
    else:
        return False

summary_factor_1["eligible"] = summary_factor_1.apply(lambda x: check_quality_factor(x), axis = 1)
summary_factor_1["sign_ortention"] = summary_factor_1.apply(lambda x: 1 if (check_sign(x["spread_mean"]) + check_sign(x["ic_mean"]) 
                                                                                      + check_sign(x["beta_mean"])) > 0 else -1 , axis = 1)

summary_factor_1

,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig,spread_mean_magnitude,eligible,sign_ortention
0,z_amihud_14d,fwd_1d,-0.001911,alpha 1%,-0.061571,alpha 1%,-0.000897,alpha 1%,19.107714,True,-1
1,z_amihud_14d,fwd_5d,-0.009236,alpha 1%,-0.101050,alpha 1%,-0.004532,alpha 1%,18.471505,True,-1
2,z_amihud_14d,fwd_14d,-0.027418,alpha 1%,-0.139387,alpha 1%,-0.013487,alpha 1%,19.584141,True,-1
3,z_amihud_30d,fwd_1d,-0.001881,alpha 1%,-0.072090,alpha 1%,-0.001102,alpha 1%,18.813110,True,-1
4,z_amihud_30d,fwd_5d,-0.009176,alpha 1%,-0.108743,alpha 1%,-0.005147,alpha 1%,18.352976,True,-1
5,z_amihud_30d,fwd_14d,-0.028051,alpha 1%,-0.159735,alpha 1%,-0.014426,alpha 1%,20.036652,True,-1
6,z_momentum_14d,fwd_1d,0.000882,not significant,-0.013561,not significant,0.000048,not significant,8.824701,False,1
7,z_momentum_14d,fwd_5d,0.005320,alpha 5%,0.010104,not significant,0.001751,not significant,10.640660,False,1
8,z_momentum_14d,fwd_14d,0.012755,alpha 5%,0.048927,alpha 5%,0.006495,alpha 5%,9.110751,False,1
9,z_momentum_30d,fwd_1d,0.000489,not significant,-0.020867,not significant,-0.000210,not significant,4.885709,False,-1


In [13]:
factor_eligible = summary_factor_1[summary_factor_1["eligible"] == True].reset_index(drop= True)
factor_eligible

,factor_id,horizon,spread_mean,sig_x,ic_mean,sig_y,beta_mean,sig,spread_mean_magnitude,eligible,sign_ortention
0,z_amihud_14d,fwd_1d,-0.001911,alpha 1%,-0.061571,alpha 1%,-0.000897,alpha 1%,19.107714,True,-1
1,z_amihud_14d,fwd_5d,-0.009236,alpha 1%,-0.101050,alpha 1%,-0.004532,alpha 1%,18.471505,True,-1
2,z_amihud_14d,fwd_14d,-0.027418,alpha 1%,-0.139387,alpha 1%,-0.013487,alpha 1%,19.584141,True,-1
3,z_amihud_30d,fwd_1d,-0.001881,alpha 1%,-0.072090,alpha 1%,-0.001102,alpha 1%,18.813110,True,-1
4,z_amihud_30d,fwd_5d,-0.009176,alpha 1%,-0.108743,alpha 1%,-0.005147,alpha 1%,18.352976,True,-1
5,z_amihud_30d,fwd_14d,-0.028051,alpha 1%,-0.159735,alpha 1%,-0.014426,alpha 1%,20.036652,True,-1
6,z_vol_14d,fwd_5d,-0.008009,alpha 1%,-0.118239,alpha 1%,-0.004323,alpha 1%,16.017134,True,-1
7,z_vol_14d,fwd_14d,-0.026430,alpha 1%,-0.148423,alpha 1%,-0.013274,alpha 1%,18.878659,True,-1
8,z_vol_30d,fwd_1d,-0.002442,alpha 1%,-0.093640,alpha 1%,-0.001361,alpha 1%,24.424604,True,-1
9,z_vol_30d,fwd_5d,-0.011169,alpha 1%,-0.138187,alpha 1%,-0.006125,alpha 1%,22.337064,True,-1


In [14]:
# chỉ lọc ra những factor đủ điều kiện
col_factor_qualified = factor_eligible["factor_id"].unique().tolist()


new_df = df[["timestamp","symbol","in_universe"]+ col_factor_qualified + col_forward_return].copy()
new_df

,timestamp,symbol,in_universe,z_amihud_14d,z_amihud_30d,z_vol_14d,z_vol_30d,z_vol_7d,z_vol_of_vol_14d,fwd_1d,fwd_5d,fwd_14d
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,-1.152049,-1.279244,-1.162525,-1.333472,-1.207511,-0.707390,0.003372,0.019895,0.227973
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,-1.136343,-1.252158,-0.921146,-0.631311,-0.961410,-0.026073,0.011316,0.055735,0.257275
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,-0.936310,-1.048271,-0.075388,-0.642522,0.864584,-1.009617,0.027662,0.016399,0.126829
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.853872,-0.953418,0.717912,0.249583,0.357615,-0.571819,0.016380,0.030426,0.203359
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,-0.952727,-1.058538,-0.569293,-0.352224,-1.212615,0.303731,0.003268,0.061491,0.212283
...,...,...,...,...,...,...,...,...,...,...,...,...
258042,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.024672,0.105572,0.146774
258043,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.012474,0.084218,0.092330
258044,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.017225,0.055387,0.042519
258045,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,NaN,NaN,NaN,NaN,0.073563,0.161755,0.094616


In [15]:
"""Thực hiện kiểm định corr cho toàn bộ những biến vừa được chọn"""
factor_eligible_1day = factor_eligible[factor_eligible["horizon"] == "fwd_1d"]


def check_corr(df, col_factor_qualified):
    in_universe = df[df["in_universe"] == 1].copy()

    daily_corr = (
        in_universe
        .groupby("timestamp")[col_factor_qualified]
        .corr(method="spearman")
        .rename_axis(["timestamp", "factor"])
    )

    median_corr = daily_corr.groupby(level="factor").median()

    upper_corr = median_corr.where(
        np.triu(np.ones(median_corr.shape), k=1).astype(bool)
    )
    
    # Chuyển matrix sang dạng factor_a, factor_b, corr
    corr_pair = (
        upper_corr
        .stack()
        .reset_index()
    )

    corr_pair.columns = ["factor_a", "factor_b", "corr"]


    return corr_pair.dropna()
corr_factor = check_corr(new_df, col_factor_qualified)
corr_factor
new = corr_factor.merge(factor_eligible_1day[["factor_id","spread_mean","ic_mean"]], left_on="factor_a", right_on="factor_id", how = 'inner' )
new = new.merge(factor_eligible_1day[["factor_id","spread_mean","ic_mean"]], left_on="factor_b", right_on="factor_id", how = 'inner')

new = new.rename(columns={"spread_mean_x": "spread_mean_a",
                     "ic_mean_x": "ic_mean_a","spread_mean_y":"spread_mean_b","ic_mean_y":"ic_mean_b" })

new.drop(columns=["factor_id_x","factor_id_y"])

,factor_a,factor_b,corr,spread_mean_a,ic_mean_a,spread_mean_b,ic_mean_b
0,z_amihud_14d,z_amihud_30d,0.971988,-0.001911,-0.061571,-0.001881,-0.072090
1,z_amihud_14d,z_vol_30d,0.544737,-0.001911,-0.061571,-0.002442,-0.093640
2,z_amihud_14d,z_vol_of_vol_14d,0.351032,-0.001911,-0.061571,-0.001642,-0.057915
3,z_amihud_30d,z_vol_30d,0.608038,-0.001881,-0.072090,-0.002442,-0.093640
4,z_amihud_30d,z_vol_of_vol_14d,0.397638,-0.001881,-0.072090,-0.001642,-0.057915
5,z_vol_30d,z_vol_of_vol_14d,0.699314,-0.002442,-0.093640,-0.001642,-0.057915


- Bỏ đi z_amihud_30d (vì speard_mean của z_amihud_14d lớn hơn)  
- đồng thời bỏ cả z_vol_7d và z_vol_14d 

In [ ]:
col_factor_qualified = ["z_amihud_14d","z_vol_30d","z_vol_of_vol_14d"]

new_df= df[["timestamp","symbol","in_universe"] + col_factor_qualified + col_forward_return]
new_df

,timestamp,symbol,in_universe,z_amihud_14d,z_vol_30d,z_vol_of_vol_14d,fwd_1d,fwd_5d,fwd_14d
0,2023-01-01 00:00:00+00:00,BTC/USDT,1,-1.152049,-1.333472,-0.707390,0.003372,0.019895,0.227973
1,2023-01-01 00:00:00+00:00,ETH/USDT,1,-1.136343,-0.631311,-0.026073,0.011316,0.055735,0.257275
2,2023-01-01 00:00:00+00:00,XRP/USDT,1,-0.936310,-0.642522,-1.009617,0.027662,0.016399,0.126829
3,2023-01-01 00:00:00+00:00,DOGE/USDT,1,-0.853872,0.249583,-0.571819,0.016380,0.030426,0.203359
4,2023-01-01 00:00:00+00:00,BNB/USDT,1,-0.952727,-0.352224,0.303731,0.003268,0.061491,0.212283
...,...,...,...,...,...,...,...,...,...
258042,2025-12-31 00:00:00+00:00,GNO/USDT,0,NaN,NaN,NaN,0.024672,0.105572,0.146774
258043,2025-12-31 00:00:00+00:00,BNT/USDT,0,NaN,NaN,NaN,0.012474,0.084218,0.092330
258044,2025-12-31 00:00:00+00:00,REQ/USDT,0,NaN,NaN,NaN,0.017225,0.055387,0.042519
258045,2025-12-31 00:00:00+00:00,GTC/USDT,0,NaN,NaN,NaN,0.073563,0.161755,0.094616


Tổng kết:
- Tất cả phần công việc phía trên cho mục đích chọn ra những Factor đủ điều kiện thống kê để từ đó việc xây dựng về sau sẽ chỉ dùng các Factor đó  
  
- Những kết quả thống kê được tập hợp từ việc:   
    - ic: information coeff có ý nghĩa thống kê hay không
    - beta: hệ số hồi quy có đủ điều kiện
    - spread: độ chênh lệch giữa nhóm z_factor_i cao và nhóm z_factor_i thấp
  

---> Sau cùng ta có các Factor đủ điều kiện cho phần sau: "z_amihud_14d","z_vol_30d","z_vol_of_vol_14d"